In [4]:
import random
import numpy as np

from reallm import Evaluation

def evaluate_pseudorandom(func):
    randseed = random.randint(1,23489)
    try:
        res = func(randseed)
        return Evaluation(
            fitness = res,
            errors = None
        )
    except Exception as e:
        return Evaluation(
            fitness = float('-inf'),
            errors = str(e)
        )

In [29]:
from reallm import LLMSolutionEngine, Solution, ProblemSpec

## PROMPT-TO-SOLUTION ENGINE
### uses api information and problem details for easy prompt evaluation

import os

engine = LLMSolutionEngine(
    api_spec=dict(
        host="api.openai.com",
        key=os.environ['OPENAI_API_KEY'],
        model="gpt-4o-mini",
        timeout=20
    ),
    problem_spec=ProblemSpec(
        description="Generate a random number (integer) from some random seed (int)",
        template="""def random_num(inp: int) -> int:\n\tpass""",
        entrypoint="random_num",
        evaluator=evaluate_pseudorandom
    )
)

In [30]:

@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def i1_solution():
    return f'''<PROBLEM_DESCR>
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>. 
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''


In [36]:

@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def e1_solution(*parent_solns):
    return f'''<PROBLEM_DESCR>
I have {len(parent_solns)} existing algorithms with their codes as follows:
{'\n'.join([f'Solution num {i}:\n{ss.program}\n' for i, ss in enumerate(parent_solns)])}
Please help me create a new algorithm that has a totally different form from the given ones. 
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''


In [37]:

@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def e2_solution(*parent_solns):
    return f'''<PROBLEM_DESCR>
I have {len(parent_solns)} existing algorithms with their codes as follows:
{'\n'.join([f'Solution num {i}:\n{ss.program}\n' for i, ss in enumerate(parent_solns)])}
Please help me create a new algorithm that has a totally different form from the given ones but can be motivated from them.
1. Firstly, identify the common backbone idea in the provided algorithms. 
2. Secondly, based on the backbone idea describe your new algorithm in one sentence. The description must be inside within boxed <<>>.
3. Thirdly, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''


In [33]:

@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def m1_solution(parent_soln):
    return f'''<PROBLEM_DESCR>
I have one algorithm with its code as follows. Algorithm description:
{parent_soln.algorithm}
Code:
{str(parent_soln.program)}
Please assist me in creating a new algorithm that has a different form but can be a modified version of the algorithm provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''


In [34]:

@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def m2_solution(parent_soln):
    return f'''<PROBLEM_DESCR>
I have one algorithm with its code as follows. Algorithm description:
{parent_soln.algorithm}
Code:
{str(parent_soln.program)}
Please identify the main algorithm parameters and assist me in creating a new algorithm that has a different parameter settings of the score function provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''


In [38]:

import numpy as np
from typing import List

def selection(k, population: List[Solution]):
    funcs = [f for f in population if not math.isinf(f.evaluation.fitness)]
    func = sorted(funcs, key=lambda f: f.evaluation.fitness, reverse=True)
    p = [1 / (r + len(func)) for r in range(len(func))]
    p = np.array(p)
    p = p / np.sum(p)
    return np.random.choice(func, p=p)


In [ ]:
from itertools import chain
from typing import List


for g_idx in range(max_gen):
    pass





In [4]:
## MAIN LOOP
### reasons solely about the prompt composition, shouldn't need to think about prompt or LLM details at all

### useful here to know about the Solution class, intended to implement all meaningful interfaces for this flow

from itertools import chain
from typing import List

def _top_k_solutions(pop: List[Solution], k: int):
    return list(sorted(pop, key=lambda s: s.fitness, reverse=True))[:k]


def funsearch(max_gen=1, initial_pop=5, num_isles=3, max_isle_pop=3):

    all_evaluations = []

    seed_population = _top_k_solutions([initial_solution() for _ in range(initial_pop)], num_isles)

    for g_idx in range(max_gen):

        islands = []
        for is_idx, seed_soln in enumerate(seed_population):
            islands.append([mutated_solution(seed_soln) for _ in range(max_isle_pop)])

        all_solns = list(chain(*islands))
        seed_population = _top_k_solutions(all_solns, num_isles)

    return _top_k_solutions(seed_population, 1).pop()

In [5]:
%%time
roar = funsearch()

CPU times: user 552 ms, sys: 12.3 ms, total: 564 ms
Wall time: 20.1 s


In [7]:
print(roar.program)

def random_num(inp: int) -> int:
    a = 22695477
    c = 1
    m = 2**32
    
    inp = (a * inp + c) % m
    return inp ^ (inp >> 16)


